# 04 · Definição das arquiteturas

Define em PyTorch as três arquiteturas comparadas, CNN 1D, BiLSTM e Transformer, todas com dois ramos (um para cada visão) e a mesma cabeça de classificação. Como entradas, saída, perda e otimizador são os mesmos, a única diferença entre os modelos é o ramo.

Este notebook não usa o `dataset.npz`: as arquiteturas são testadas com tensores aleatórios nas dimensões reais (Global 2.001, Local 201). Os hiperparâmetros definidos aqui são os iniciais; os finais saem da busca do notebook 05b.

In [ ]:
import torch
import torch.nn as nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch:', torch.__version__, '| device:', device)

GLOBAL_LEN = 2001
LOCAL_LEN  = 201

PyTorch: 2.11.0+cpu | device: cpu


## Estrutura comum

Cada arquitetura fornece um ramo, um módulo que transforma uma visão de forma `(B, 1, L)` em um vetor de atributos `(B, out_dim)`. A classe `DoisRamos` cria dois ramos independentes, um para cada visão, concatena seus atributos e aplica uma camada densa de 64 unidades com *dropout* e uma saída única. A saída é o *logit*; a sigmoide é aplicada pela função de perda (`BCEWithLogitsLoss`).

In [ ]:
class DoisRamos(nn.Module):
    """Une um ramo Global e um Local (mesma classe) e classifica."""
    def __init__(self, fabrica_ramo, p_dropout=0.3):
        super().__init__()
        self.ramo_global = fabrica_ramo()
        self.ramo_local  = fabrica_ramo()
        d = self.ramo_global.out_dim
        self.cabeca = nn.Sequential(
            nn.Linear(2 * d, 64), nn.ReLU(), nn.Dropout(p_dropout),
            nn.Linear(64, 1),
        )

    def forward(self, g, l):        # g:(B,1,2001)  l:(B,1,201)
        fg = self.ramo_global(g)    # (B, d)
        fl = self.ramo_local(l)     # (B, d)
        z = torch.cat([fg, fl], dim=1)
        return self.cabeca(z).squeeze(-1)   # (B,) logits

## 1. CNN 1D

Blocos de convolução 1D, ReLU e *max-pooling*, como em Shallue e Vanderburg (2018). O *pooling* máximo global no final torna o ramo independente do comprimento da entrada, de modo que a mesma classe serve para as duas visões.

In [ ]:
class RamoCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 16, 5, padding=2), nn.ReLU(), nn.MaxPool1d(5, 2),
            nn.Conv1d(16, 32, 5, padding=2), nn.ReLU(), nn.MaxPool1d(5, 2),
            nn.Conv1d(32, 64, 5, padding=2), nn.ReLU(),
            nn.AdaptiveMaxPool1d(1),
        )
        self.out_dim = 64

    def forward(self, x):            # (B,1,L)
        return self.net(x).squeeze(-1)   # (B,64)

## 2. BiLSTM

Camadas LSTM bidirecionais, que percorrem a curva dobrada nos dois sentidos. A saída do ramo é a média dos estados ocultos ao longo da sequência.

In [ ]:
class RamoBiLSTM(nn.Module):
    def __init__(self, hidden=64, camadas=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden, num_layers=camadas,
                            batch_first=True, bidirectional=True, dropout=0.1)
        self.out_dim = 2 * hidden

    def forward(self, x):            # (B,1,L)
        x = x.transpose(1, 2)        # (B,L,1)
        saida, _ = self.lstm(x)      # (B,L,2*hidden)
        return saida.mean(dim=1)     # (B,2*hidden)

## 3. Transformer

*Embedding* convolucional, codificação posicional senoidal e camadas de autoatenção multicabeça, com base em Salinas et al. (2023).

O custo da autoatenção cresce com o quadrado do comprimento da sequência. Por isso, o *embedding* usa uma convolução com passo 8, que reduz a Visão Global de 2.001 para 251 posições antes da atenção e torna o treino viável no Colab.

In [ ]:
def codificacao_posicional(comprimento, dim, device):
    """Codificacao posicional senoidal (Vaswani et al., 2017)."""
    pos = torch.arange(comprimento, device=device).unsqueeze(1).float()
    i = torch.arange(dim, device=device).unsqueeze(0).float()
    ang = pos / torch.pow(torch.tensor(10000.0, device=device), (2 * (i // 2)) / dim)
    pe = torch.zeros(comprimento, dim, device=device)
    pe[:, 0::2] = torch.sin(ang[:, 0::2])
    pe[:, 1::2] = torch.cos(ang[:, 1::2])
    return pe.unsqueeze(0)            # (1, comprimento, dim)


class RamoTransformer(nn.Module):
    def __init__(self, d_model=64, nhead=4, camadas=2, stride=8):
        super().__init__()
        # embedding convolucional com passo: reduz o comprimento antes da atencao
        self.embed = nn.Conv1d(1, d_model, kernel_size=2 * stride + 1,
                               stride=stride, padding=stride)
        camada = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=128,
                                            dropout=0.1, batch_first=True)
        self.encoder = nn.TransformerEncoder(camada, camadas)
        self.out_dim = d_model

    def forward(self, x):            # (B,1,L)
        z = self.embed(x)            # (B,d,L')
        z = z.transpose(1, 2)        # (B,L',d)
        z = z + codificacao_posicional(z.size(1), z.size(2), z.device)
        z = self.encoder(z)          # (B,L',d)
        return z.mean(dim=1)         # (B,d)

## Teste

Passa um lote aleatório pelas três arquiteturas, confere que a saída tem forma `(B,)`, um *logit* por exemplo, e conta os parâmetros de cada uma.

In [ ]:
B = 8
g = torch.randn(B, 1, GLOBAL_LEN, device=device)
l = torch.randn(B, 1, LOCAL_LEN, device=device)

arquiteturas = {
    'CNN 1D':      RamoCNN,
    'BiLSTM':      RamoBiLSTM,
    'Transformer': RamoTransformer,
}

for nome, fabrica in arquiteturas.items():
    modelo = DoisRamos(fabrica).to(device)
    with torch.no_grad():
        saida = modelo(g, l)
    n_par = sum(p.numel() for p in modelo.parameters())
    print(f'{nome:12s}  saida={tuple(saida.shape)}  parametros={n_par:,}')

CNN 1D        saida=(8,)  parametros=34,305
BiLSTM        saida=(8,)  parametros=283,777
Transformer   saida=(8,)  parametros=144,513


## Próximo passo

O notebook 05 treina as três arquiteturas com a configuração inicial.